In [1]:
import pandas as pd

merged_df = pd.read_csv(
    "../dataset/merged_trader_sentiment.csv",
    parse_dates=['date']
)

trader_summary = (
    merged_df
    .groupby('Account')
    .agg(
        total_trades=('Closed PnL', 'count'),
        total_pnl=('Closed PnL', 'sum'),
        avg_pnl=('Closed PnL', 'mean'),
        win_rate=('is_profitable', 'mean'),
        avg_trade_size=('Size USD', 'mean'),
        pnl_volatility=('Closed PnL', 'std')
    )
    .reset_index()
)

trader_summary['win_rate'] *= 100
trader_summary.head()

,Account,total_trades,total_pnl,avg_pnl,win_rate,avg_trade_size,pnl_volatility
0,0x083384f897ee0f19899168e3b1bec365f52a9012,2486,965588.745995,388.410598,40.305712,16277.368089,3759.937556
1,0x23e7a7f8d14b550961925fbfdaa92f5d195ba5bd,1192,14102.674084,11.831102,40.016779,2019.281124,121.625425
2,0x271b280974205ca63b716753467d5a371de622ab,4,208.817460,52.204365,75.000000,39530.987500,61.502017
3,0x28736f43f1e871e6aa8b1148d38d4994275d72c4,2737,58534.883527,21.386512,47.826087,372.626975,133.223824
4,0x2c229d22b100a7beb69122eed721cee9b24011dd,378,43040.784276,113.864509,51.851852,2914.655714,519.038684


In [2]:
top_10_pct = trader_summary['total_pnl'].quantile(0.90)
bottom_10_pct = trader_summary['total_pnl'].quantile(0.10)

top_traders = trader_summary[trader_summary['total_pnl'] >= top_10_pct]
bottom_traders = trader_summary[trader_summary['total_pnl'] <= bottom_10_pct]

top_traders.shape, bottom_traders.shape

((4, 7), (4, 7))

In [3]:
comparison = pd.DataFrame({
    'metric': [
        'Avg Trades',
        'Avg Win Rate (%)',
        'Avg Trade Size',
        'Avg PnL',
        'PnL Volatility'
    ],
    'Top Traders': [
        top_traders['total_trades'].mean(),
        top_traders['win_rate'].mean(),
        top_traders['avg_trade_size'].mean(),
        top_traders['avg_pnl'].mean(),
        top_traders['pnl_volatility'].mean()
    ],
    'Bottom Traders': [
        bottom_traders['total_trades'].mean(),
        bottom_traders['win_rate'].mean(),
        bottom_traders['avg_trade_size'].mean(),
        bottom_traders['avg_pnl'].mean(),
        bottom_traders['pnl_volatility'].mean()
    ]
})

comparison

,metric,Top Traders,Bottom Traders
0,Avg Trades,3995.750000,815.750000
1,Avg Win Rate (%),46.115874,14.950498
2,Avg Trade Size,6939.729075,4858.843891
3,Avg PnL,196.556064,-3.871502
4,PnL Volatility,1642.784091,166.453238


In [4]:
merged_with_segment = merged_df.merge(
    trader_summary[['Account', 'total_pnl']],
    on='Account',
    how='left'
)

merged_with_segment['segment'] = 'Middle'
merged_with_segment.loc[
    merged_with_segment['total_pnl'] >= top_10_pct, 'segment'
] = 'Top'
merged_with_segment.loc[
    merged_with_segment['total_pnl'] <= bottom_10_pct, 'segment'
] = 'Bottom'

In [5]:
segment_sentiment = (
    merged_with_segment
    .groupby(['segment', 'sentiment'])
    .size()
    .reset_index(name='trade_count')
)

segment_sentiment

,segment,sentiment,trade_count
0,Bottom,Extreme Fear,348
1,Bottom,Extreme Greed,573
2,Bottom,Fear,1256
3,Bottom,Greed,982
4,Bottom,Neutral,104
5,Middle,Extreme Fear,721
6,Middle,Extreme Greed,2338
7,Middle,Fear,5908
8,Middle,Greed,6569
9,Middle,Neutral,1082


In [6]:
trader_summary.to_csv(
    "../outputs/summary_tables/trader_level_summary.csv",
    index=False
)

comparison.to_csv(
    "../outputs/summary_tables/top_vs_bottom_traders.csv",
    index=False
)

- Top-performing traders exhibit significantly higher win rates and lower PnL volatility,
  indicating disciplined risk management.
- Profitable traders do not necessarily trade more frequently; instead, they focus on
  consistent execution with controlled trade sizes.
- Bottom-performing traders show higher volatility and poorer performance during
  Extreme Fear and Extreme Greed phases, suggesting emotionally driven decision-making.
- Avoidance of extreme sentiment regimes appears to be a common trait among
  consistently profitable traders.

- Reduce position sizes during Extreme Fear and Extreme Greed to control downside risk.
- Favor disciplined, lower-volatility trading approaches over high-frequency aggression.
- Incorporate sentiment-based risk filters to dynamically adjust exposure.
- Use loss-per-dollar metrics to detect inefficient risk behavior early.